# C.5 花更多計算值得嗎？

# C：推理步驟、驗證與測試時算力

前置：下一字生成、SFT與標準答案驗證。像解數學題：寫出步驟、多試幾種辦法、檢查答案，是三個不同動作。真實推理質量需要trained模型；默認例子用可驗證的數字，避免把長文字當思考保證。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：多試幾次，還要知道怎麼選**

候選包含正解，不代表最後答案已經選中。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/reasoning.svg")))

**先想一想：**8個短答與1個長答，tokens相同延遲一定相同嗎？

多候選和長步驟都用預算。固定候選數或總生成tokens比較準確率，延遲也一起量；更多計算不保證回報一直增加。

**把直覺寫成數學：**budget=N×average_generated_tokens+verification_cost；實際延遲可受並行影響。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
plans = [{"samples": 1, "tokens_each": 80}, {"samples": 4, "tokens_each": 20}]
for plan in plans:
    print(plan, "生成token預算", plan["samples"] * plan["tokens_each"])
print("真實實驗再記錄accuracy、coverage、selection與wall time")

**如何讀結果：**沒有僞造質量曲線；不同並行和cache讓相同tokens有不同執行成本。

**只改一件事：**只固定總tokens而改變候選數。
